# v4 Phase 0 — Diagnostics on the v3 models (read-only)

Nothing is trained and no v3 file is written. Questions come from `sessions/2026-10-12.md` (Sections 9–16);
the plan, verified by Manish's Claude, is `PLAN-after-review.md`.

**Methods (full names; the v3 code keys are in brackets):** Fixed-weight MAPPO (α = 0.5) [`B4`];
Hand-coded τ-rule MAPPO (α = 0.1 + 0.8·τ/3) [`H`]; MAPPO + PAH (τ-rule + learned residual) [`M`];
MAPPO + PAH with Hungarian assignment turned off [`M-noHungarian`].
Stage 2 = 5 drones, 5 obstacles (seeds 42–46). Stage 3 = 8 drones, 10 obstacles (seed 42).

| Check | Question |
|---|---|
| Collision-type check | Are the collisions drone–drone or drone–obstacle? |
| Safety-reward frequency check | How often is the safety *reward* switched on? |
| α spread check | How much does α actually move, and what drives it? |
| IGAT-MARL metrics check | Loss-of-separation and active conflict edges |
| Advantage-correlation check | Does α's mixing of the mission and safety advantages actually change the actor's learning signal? |

**Passes.**
- *Evaluation pass* (collision-type, IGAT-MARL metrics): all 18 final models, deterministic policy,
  200 episodes, env seed 7001 — same protocol and episode layouts as `v3/results-table`.
- *Training-like replay* (safety-reward frequency, α spread): every saved checkpoint of every run
  (Stage 2: 5 seeds per method; Stage 3: seed 42), stochastic policy, 50 episodes, env seed 9101.
  **This is an approximation of training conditions** — saved checkpoints replayed on a new seed, not the
  actual training rollouts.
- *Advantage pass*: training-style batches (512 env steps, stochastic, success bonus 20 as in training),
  advantages computed with the verbatim v3 `RolloutBuffer` and each checkpoint's own critic.

**Definitions.** Danger radius = 3 × collision radius = 9 units (the radius the v3 reward uses for the
safety reward and the conflict graph uses for an edge). Loss-of-separation = a drone pair closer than 9
units at a step. In v3 training the actor receives `α·z(A_mission) + (1−α)·z(A_safety)`, where both
advantages use the single critic trained on the combined reward and are z-normalized separately; so the
safety advantage is non-zero on most steps even when the safety reward is zero.

**Code.** `../common/v3_env.py` (frozen, verified copy of the v3 env + networks + RolloutBuffer),
`../common/v3_registry.py` (all 18 models), `diag.py` (instrumented rollout, advantage batch).

In [1]:
# Cell 1 -- setup
import os, sys, json, time, math
import numpy as np
import pandas as pd
import torch
torch.set_num_threads(1)

sys.path.insert(0, os.path.abspath('../common'))
import v3_registry as R
import diag as D

OUT = 'output'
os.makedirs(OUT, exist_ok=True)

N_EVAL, EVAL_SEED = 200, 7001      # same as v3/results-table
N_REPLAY, REPLAY_SEED = 50, 9101   # training-like replay
N_ADV_BATCHES, ADV_SEED0 = 10, 9201

# every run used in the replay and advantage passes: Stage 2 all 5 seeds, Stage 3 seed 42
RUNS = [(2, m, s) for m in ('B4', 'H', 'M') for s in (42, 43, 44, 45, 46)] + \
       [(3, m, 42) for m in ('B4', 'M', 'M-noHungarian')]

# full names for every human-facing label (no short codes)
NAME = {'B4': 'Fixed-weight MAPPO', 'H': 'Hand-coded τ-rule MAPPO', 'M': 'MAPPO + PAH',
        'M-noHungarian': 'MAPPO + PAH, Hungarian off'}
NAME_WRAP = {'B4': 'Fixed-weight\nMAPPO', 'H': 'Hand-coded\nτ-rule MAPPO', 'M': 'MAPPO + PAH',
             'M-noHungarian': 'MAPPO + PAH,\nHungarian off'}
STAGE = {2: 'Stage 2 (5 drones, 5 obstacles)', 3: 'Stage 3 (8 drones, 10 obstacles)'}
ORDER = {2: ['B4', 'H', 'M'], 3: ['B4', 'M', 'M-noHungarian']}
print(R.check_all())

18 models, all final_model.pt found


In [2]:
# Cell 2 -- GATE: the frozen copy must reproduce v3 episode-by-episode, or stop here
RAW_V3 = os.path.join(R.V3, 'results-table', 'output', 'raw')

def outcomes(entry, n=N_EVAL, seed=EVAL_SEED):
    actor, pah = D.load_checkpoint(R.final_model(entry))
    env = D.make_env(entry, seed)
    return [D.run_episode(env, actor, pah, entry['method'], 'det')[0]['outcome'] for _ in range(n)]

for stage, method, seed in [(3, 'M', 42), (2, 'B4', 43)]:
    e = R.find(stage, method, seed)
    mine = outcomes(e)
    ref = [r['outcome'] for r in json.load(open('{}/s{}_{}_seed{}.json'.format(RAW_V3, stage, method, seed)))['records']]
    same = sum(a == b for a, b in zip(mine, ref))
    print('{}, {}, seed {}: {}/{} episodes identical to v3'.format(STAGE[stage], NAME[method], seed, same, len(ref)))
    assert same == len(ref), 'GATE FAILED -- do not trust anything below'
print('GATE PASSED')

Stage 3 (8 drones, 10 obstacles), MAPPO + PAH, seed 42: 200/200 episodes identical to v3


Stage 2 (5 drones, 5 obstacles), Fixed-weight MAPPO, seed 43: 200/200 episodes identical to v3
GATE PASSED


In [3]:
# Cell 3 -- Evaluation pass (collision-type, IGAT-MARL metrics): all 18 final models, deterministic
EVAL_CSV = OUT + '/eval_episodes.csv'
if os.path.exists(EVAL_CSV):
    ev = pd.read_csv(EVAL_CSV, keep_default_na=False)
    print('loaded cached', EVAL_CSV)
else:
    rows = []
    for e in R.REGISTRY:
        t0 = time.time()
        actor, pah = D.load_checkpoint(R.final_model(e))
        env = D.make_env(e, EVAL_SEED)
        for k in range(N_EVAL):
            s, _ = D.run_episode(env, actor, pah, e['method'], 'det')
            if s['outcome'] == 'collision':
                assert env._any_collision() and s['collision_kind'] != 'unknown'   # independent check
            rows.append(dict(stage=e['stage'], method=e['method'], seed=e['seed'], episode=k, **s))
        print('{} | {:28s} seed {} [{:.0f}s]'.format(STAGE[e['stage']], NAME[e['method']], e['seed'], time.time() - t0))
    ev = pd.DataFrame(rows)
    ev.to_csv(EVAL_CSV, index=False)
print(len(ev), 'episodes')

loaded cached output/eval_episodes.csv
3600 episodes


In [4]:
# Cell 4 -- Training-like replay (safety-reward frequency, α spread): every checkpoint, stochastic
REPLAY_CSV = OUT + '/training_like_replay.csv'
if os.path.exists(REPLAY_CSV):
    tr = pd.read_csv(REPLAY_CSV)
    print('loaded cached', REPLAY_CSV)
else:
    rows = []
    for stage, method, seed in RUNS:
        e = R.find(stage, method, seed)
        ckpts = R.checkpoints(e) + [(8000, R.final_model(e))]       # final model plotted at episode 8000
        t0 = time.time()
        for idx, (ep, path) in enumerate(ckpts):
            actor, pah = D.load_checkpoint(path)
            env = D.make_env(e, REPLAY_SEED)
            gen = torch.Generator().manual_seed(REPLAY_SEED)
            S, A = [], []
            for _ in range(N_REPLAY):
                s, a = D.run_episode(env, actor, pah, method, 'stoch', gen)
                S.append(s); A.append(a)
            S = pd.DataFrame(S); A = np.concatenate(A)
            w = S['steps'] * e['n_drones']                              # drone-step weights, like a batch
            avg = lambda col: float(np.average(S[col], weights=w))
            rows.append(dict(
                stage=stage, method=method, seed=seed, ckpt_index=idx, episode=ep, final=(idx == len(ckpts) - 1),
                success=100 * (S.outcome == 'success').mean(), collision=100 * (S.outcome == 'collision').mean(),
                safety_pct=100 * avg('safety_frac'), safety_drone_pct=100 * avg('safety_drone_frac'),
                safety_obs_pct=100 * avg('safety_obs_frac'), conflict_pct=100 * avg('conflict_frac'),
                alpha_mean=float(A.mean()), alpha_p10=float(np.percentile(A, 10)), alpha_p50=float(np.percentile(A, 50)),
                alpha_p90=float(np.percentile(A, 90)), alpha_lt_05_pct=100 * float((A < 0.5).mean()),
                alpha_lt_08_pct=100 * float((A < 0.8).mean()), drone_steps=int(w.sum())))
        print('{} | {:28s} seed {}: {} checkpoints [{:.0f}s]'.format(STAGE[stage], NAME[method], seed, len(ckpts), time.time() - t0))
    tr = pd.DataFrame(rows)
    tr.to_csv(REPLAY_CSV, index=False)
print(len(tr), 'checkpoint rows')

loaded cached output/training_like_replay.csv
285 checkpoint rows


In [5]:
# Cell 5 -- What drives α when there is no conflict? (final MAPPO + PAH models)
# Moved here from a one-off check (session 2026-10-12, Section 12.3) so it is reproducible.
ALPHA_CSV = OUT + '/alpha_vs_target_distance.csv'
if os.path.exists(ALPHA_CSV):
    ad = pd.read_csv(ALPHA_CSV)
    print('loaded cached', ALPHA_CSV)
else:
    rows = []
    for stage, method, seed in [r for r in RUNS if r[1] in ('M', 'M-noHungarian')]:
        e = R.find(stage, method, seed)
        actor, pah = D.load_checkpoint(R.final_model(e))
        env = D.make_env(e, REPLAY_SEED)
        gen = torch.Generator().manual_seed(REPLAY_SEED)
        al, dt, nc = [], [], []
        for _ in range(N_REPLAY):
            obs = env.reset(); done = False
            while not done:
                obs, _, te, trn, info = env.step(D.stochastic_action(actor, obs, gen))
                p = info['pah_inputs']
                al.append(D.pah_alpha(pah, p, e['n_drones'])); dt.append(p['d_target']); nc.append(p['n_conflict'])
                done = te or trn
        al, dt, nc = map(np.concatenate, (al, dt, nc))
        free = nc == 0
        bins = [(0, 50), (50, 150), (150, 300), (300, 1000)]
        med = {'median_alpha_d{}_{}'.format(lo, hi): float(np.median(al[free & (dt >= lo) & (dt < hi)]))
               if (free & (dt >= lo) & (dt < hi)).any() else float('nan') for lo, hi in bins}
        rows.append(dict(stage=stage, method=method, seed=seed, drone_steps=len(al),
                         no_conflict_pct=100 * free.mean(), alpha_min=float(al[free].min()), alpha_max=float(al[free].max()),
                         corr_alpha_dtarget=float(np.corrcoef(al[free], dt[free])[0, 1]), **med))
    ad = pd.DataFrame(rows)
    ad.to_csv(ALPHA_CSV, index=False)
show = ad.assign(method=ad.method.map(NAME))
print(show.round(3).to_string(index=False))

loaded cached output/alpha_vs_target_distance.csv
 stage                     method  seed  drone_steps  no_conflict_pct  alpha_min  alpha_max  corr_alpha_dtarget  median_alpha_d0_50  median_alpha_d50_150  median_alpha_d150_300  median_alpha_d300_1000
     2                MAPPO + PAH    42        11415           99.982        0.7        0.9              -0.728               0.883                 0.900                  0.733                     0.7
     2                MAPPO + PAH    43        12220          100.000        0.7        0.9              -0.878               0.900                 0.900                  0.700                     0.7
     2                MAPPO + PAH    44        11775           99.966        0.7        0.9              -0.892               0.900                 0.900                  0.700                     0.7
     2                MAPPO + PAH    45        11880           99.983        0.7        0.9              -0.327               0.785               

In [6]:
# Cell 6 -- Advantage-correlation check (PLAN Step 2, revised per Manish's review; no thresholds)
ADV_CSV = OUT + '/advantage_correlation.csv'

def adv_ckpts(stage, method, seed):
    e = R.find(stage, method, seed)
    if stage == 2:
        return e, [('final', R.final_model(e))]
    c = R.checkpoints(e)                                        # Stage 3: early / middle / late + final
    pick = [c[0], c[len(c) // 2], c[-1]]
    return e, [('ep{}'.format(ep), p) for ep, p in pick] + [('final', R.final_model(e))]

if os.path.exists(ADV_CSV):
    adv = pd.read_csv(ADV_CSV)
    print('loaded cached', ADV_CSV)
else:
    # GATE: Fixed-weight MAPPO has constant α = 0.5, so 0.5·A_mission + 0.5·A_safety must equal the
    # combined-reward advantage to float precision. If not, the extraction or batch building is wrong.
    for st in (2, 3):
        e = R.find(st, 'B4', 42)
        g = D.advantage_batch(e, R.final_model(e), ADV_SEED0, ADV_SEED0)['identity_rel_err']
        print('gate, {}, Fixed-weight MAPPO: relative error {:.1e}'.format(STAGE[st], g))
        assert g < 1e-5, 'ADVANTAGE GATE FAILED'
    rows = []
    for stage, method, seed in RUNS:
        e, cks = adv_ckpts(stage, method, seed)
        for tag, path in cks:
            for k in range(N_ADV_BATCHES):
                r = D.advantage_batch(e, path, ADV_SEED0 + k, ADV_SEED0 + k)
                rows.append(dict(stage=stage, method=method, seed=seed, checkpoint=tag, batch=k, **r))
        print('{} | {:28s} seed {} done'.format(STAGE[stage], NAME[method], seed))
    adv = pd.DataFrame(rows)
    adv.to_csv(ADV_CSV, index=False)
print(len(adv), 'batches')

loaded cached output/advantage_correlation.csv
270 batches


In [7]:
# Cell 7 -- Tables (all labels use full names)
pd.set_option('display.width', 200)
def pm(x):   # mean ± std across seeds (Stage 2) or a single value (Stage 3)
    x = np.asarray(x, dtype=float)
    return '{:.2f} ± {:.2f}'.format(x.mean(), x.std(ddof=1)) if len(x) > 1 else '{:.2f}'.format(x[0])

# Collision-type check
t1 = []
for st in (2, 3):
    for m in ORDER[st]:
        c = ev[(ev.stage == st) & (ev.method == m) & (ev.outcome == 'collision')]
        n = len(c)
        t1.append({'Stage': STAGE[st], 'Method': NAME[m], 'Collision episodes': n,
                   'Drone–obstacle': int((c.collision_kind == 'drone-obstacle').sum()),
                   'Drone–drone': int((c.collision_kind == 'drone-drone').sum()), 'Both': int((c.collision_kind == 'both').sum()),
                   '% obstacle': round(100 * (c.collision_kind != 'drone-drone').mean(), 1) if n else float('nan'),
                   '% near goal': round(100 * c.collision_near_goal.astype(bool).mean(), 1) if n else float('nan')})
T1 = pd.DataFrame(t1)

# IGAT-MARL metrics check
t4 = []
for st in (2, 3):
    for m in ORDER[st]:
        x = ev[(ev.stage == st) & (ev.method == m)]
        if st == 2:
            g = x.groupby('seed')[['los_pair_steps', 'edges_mean']].mean()
            los, edg = '{:.3f} ± {:.3f}'.format(g.los_pair_steps.mean(), g.los_pair_steps.std(ddof=1)), \
                       '{:.4f} ± {:.4f}'.format(g.edges_mean.mean(), g.edges_mean.std(ddof=1))
        else:
            ci = lambda v: 1.96 * v.std(ddof=1) / math.sqrt(len(v))
            los = '{:.3f} ± {:.3f}'.format(x.los_pair_steps.mean(), ci(x.los_pair_steps))
            edg = '{:.4f} ± {:.4f}'.format(x.edges_mean.mean(), ci(x.edges_mean))
        t4.append({'Stage': STAGE[st], 'Method': NAME[m], 'Loss-of-separation pair-steps per episode': los,
                   'Active conflict edges per step': edg})
T4 = pd.DataFrame(t4)

# Safety-reward frequency and α spread (final checkpoint; Stage 2 = mean ± std over 5 seeds)
t23 = []
for st in (2, 3):
    for m in ORDER[st]:
        f = tr[(tr.stage == st) & (tr.method == m) & (tr.final)]
        allck = tr[(tr.stage == st) & (tr.method == m)]
        t23.append({'Stage': STAGE[st], 'Method': NAME[m],
                    'Safety reward on, % drone-steps (range over training)': '{:.2f}–{:.2f}'.format(allck.safety_pct.min(), allck.safety_pct.max()),
                    'final: drone within 9': pm(f.safety_drone_pct), 'final: obstacle within 9': pm(f.safety_obs_pct),
                    'Conflict flagged, % (final)': pm(f.conflict_pct),
                    'α median (final)': pm(f.alpha_p50), 'α 10th pct (final)': pm(f.alpha_p10),
                    '% steps α < 0.5 (final)': pm(f.alpha_lt_05_pct)})
T23 = pd.DataFrame(t23)

# Advantage-correlation check (final checkpoints; Stage 2 = mean ± std over the 5 seed means)
ta = []
fin = adv[adv.checkpoint == 'final']
for st in (2, 3):
    for m in ORDER[st]:
        x = fin[(fin.stage == st) & (fin.method == m)].groupby('seed').mean(numeric_only=True)
        ta.append({'Stage': STAGE[st], 'Method': NAME[m], 'α mean': pm(x.alpha_mean),
                   'corr(mission adv., safety adv.)': pm(x.corr_m_s),
                   'raw std mission adv.': pm(x.std_m_raw), 'raw std safety adv.': pm(x.std_s_raw),
                   'cosine(own-α signal, α=0.5 signal)': pm(x.cos_own_05), 'magnitude own / α=0.5': pm(x.norm_own_over_05),
                   'corr(own-α signal, plain combined advantage)': pm(x.corr_own_comb)})
TA = pd.DataFrame(ta)
TA3 = adv[adv.stage == 3].groupby(['method', 'checkpoint'], sort=False)[
    ['alpha_mean', 'corr_m_s', 'cos_own_05', 'norm_own_over_05', 'corr_own_comb']].agg(['mean', 'std']).round(3)

for title, T in (('Collision-type check (evaluation pass)', T1), ('IGAT-MARL metrics check (evaluation pass)', T4),
                 ('Safety-reward frequency and α spread (training-like replay)', T23),
                 ('Advantage-correlation check (final checkpoints, 10 batches each)', TA)):
    print('\n' + title); print(T.to_string(index=False))
print('\nAdvantage-correlation check across Stage 3 training (mean, std over 10 batches)'); print(TA3.to_string())

with open(OUT + '/tables.md', 'w') as fh:
    for title, T in (('Collision-type check (evaluation pass)', T1), ('IGAT-MARL metrics check (evaluation pass)', T4),
                     ('Safety-reward frequency and α spread (training-like replay; Stage 2 = mean ± std over 5 seeds)', T23),
                     ('Advantage-correlation check (final checkpoints; Stage 2 = mean ± std over 5 seeds)', TA)):
        fh.write('## ' + title + '\n\n' + T.to_markdown(index=False) + '\n\n')
    fh.write('## Advantage-correlation check across Stage 3 training\n\n' + TA3.to_markdown() + '\n')
print('\nsaved', OUT + '/tables.md')


Collision-type check (evaluation pass)
                           Stage                     Method  Collision episodes  Drone–obstacle  Drone–drone  Both  % obstacle  % near goal
 Stage 2 (5 drones, 5 obstacles)         Fixed-weight MAPPO                 108             107            1     0        99.1         20.4
 Stage 2 (5 drones, 5 obstacles)    Hand-coded τ-rule MAPPO                 125             124            1     0        99.2         20.8
 Stage 2 (5 drones, 5 obstacles)                MAPPO + PAH                 105             104            1     0        99.0         21.9
Stage 3 (8 drones, 10 obstacles)         Fixed-weight MAPPO                  53              50            3     0        94.3         30.2
Stage 3 (8 drones, 10 obstacles)                MAPPO + PAH                  56              56            0     0       100.0         39.3
Stage 3 (8 drones, 10 obstacles) MAPPO + PAH, Hungarian off                  97              78           19     0      

In [8]:
# Cell 8 -- Figures (white background, full names, same house style as v3/results-table)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white',
    'font.family': 'DejaVu Sans', 'font.size': 9.5, 'axes.titlesize': 10.5, 'axes.labelsize': 9.5,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'axes.axisbelow': True,
    'grid.alpha': 0.25})
MCOL = {'B4': '#808080', 'H': '#D98E2B', 'M': '#3B6FB6', 'M-noHungarian': '#C0392B'}   # = v3 fig6 colours
KCOL = {'drone-obstacle': '#2a78d6', 'drone-drone': '#eb6834', 'both': '#1baf7a'}      # validated categorical

def save(fig, name):
    fig.savefig('{}/{}.png'.format(OUT, name), dpi=200, bbox_inches='tight')
    fig.savefig('{}/{}.svg'.format(OUT, name), bbox_inches='tight')
    plt.close(fig)

# ---- Collision-type check ------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
for ax, st in zip(axes, (2, 3)):
    sub = T1[T1.Stage == STAGE[st]]
    x = np.arange(len(sub)); bottom = np.zeros(len(sub))
    for col, kind in (('Drone–obstacle', 'drone-obstacle'), ('Drone–drone', 'drone-drone'), ('Both', 'both')):
        v = sub[col].values
        ax.bar(x, v, 0.55, bottom=bottom, color=KCOL[kind], edgecolor='white', linewidth=2, label=col)
        bottom += v
    for xi, tot, pct in zip(x, sub['Collision episodes'].values, sub['% obstacle'].values):
        ax.text(xi, tot + bottom.max() * 0.02, '{} collisions\n{:.0f}% hit an obstacle'.format(tot, pct), ha='center', va='bottom', fontsize=8.3)
    ax.set_xticks(x); ax.set_xticklabels([NAME_WRAP[m] for m in ORDER[st]], fontsize=8.5)
    ax.set_ylim(0, bottom.max() * 1.32); ax.set_ylabel('Collision episodes')
    ax.set_title('{} — {}'.format(STAGE[st], '5 seeds pooled, 1000 episodes per method' if st == 2 else 'seed 42, 200 episodes'))
axes[0].legend(loc='upper left', frameon=False, fontsize=8.5)
fig.suptitle('Collision-type check — what did the drones hit? (evaluation pass, deterministic policy)', fontsize=10.5)
plt.tight_layout(rect=(0, 0, 1, 0.94)); save(fig, 'collision_type_check')

# ---- Safety-reward frequency check ------------------------------------------------
def by_ckpt(df):   # Stage 2: mean ± std over seeds at each checkpoint index
    g = df.groupby('ckpt_index')
    return g.episode.mean(), g

fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.2), sharex=True)
for col, st in enumerate((2, 3)):
    for m in ORDER[st]:
        d = tr[(tr.stage == st) & (tr.method == m)]
        x, g = by_ckpt(d)
        for row, metric in enumerate(('safety_pct', 'conflict_pct')):
            mean = g[metric].mean()
            axes[row, col].plot(x, mean, color=MCOL[m], lw=2, marker='o', ms=3.5, label=NAME[m])
            if st == 2:
                sd = g[metric].std(ddof=1)
                axes[row, col].fill_between(x, mean - sd, mean + sd, color=MCOL[m], alpha=0.15, lw=0)
    axes[0, col].set_title(STAGE[st] + (' — mean ± std, 5 seeds' if st == 2 else ' — seed 42'))
    axes[1, col].set_xlabel('Training episode of the checkpoint')
    axes[0, col].legend(frameon=False, fontsize=8.5)
axes[0, 0].set_ylabel('% of drone-steps with the\nsafety reward on')
axes[1, 0].set_ylabel('% of drone-steps the conflict graph\nflags (PAH sees a conflict)')
for ax in axes.ravel(): ax.set_ylim(bottom=0)
fig.suptitle('Safety-reward frequency check (training-like replay: saved checkpoints, stochastic policy, 50 episodes each)', fontsize=10.5)
plt.tight_layout(rect=(0, 0, 1, 0.95)); save(fig, 'safety_reward_frequency_check')

# ---- α spread check ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4), sharey=True)
for ax, st in zip(axes, (2, 3)):
    for m in ORDER[st]:
        if m == 'B4': continue
        d = tr[(tr.stage == st) & (tr.method == m)]
        x, g = by_ckpt(d)
        ax.fill_between(x, g.alpha_p10.mean(), g.alpha_p90.mean(), color=MCOL[m], alpha=0.18, lw=0)
        ax.plot(x, g.alpha_p50.mean(), color=MCOL[m], lw=2, marker='o', ms=3.5, label=NAME[m] + ' — median, band 10th–90th pct')
    ax.axhline(0.5, color=MCOL['B4'], lw=1.6, ls='--', label='Fixed-weight MAPPO: α = 0.5')
    ax.set_ylim(0, 1); ax.set_xlabel('Training episode of the checkpoint')
    ax.set_title(STAGE[st] + (' — averaged over 5 seeds' if st == 2 else ' — seed 42'))
    ax.legend(frameon=False, fontsize=8, loc='lower right')
axes[0].set_ylabel('α over all drone-steps')
fig.suptitle('α spread check — how much does α actually move? (training-like replay)', fontsize=10.5)
plt.tight_layout(rect=(0, 0, 1, 0.93)); save(fig, 'alpha_spread_check')

# ---- IGAT-MARL metrics check -------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, metric, ylab, fmt in ((axes[0], 'los_pair_steps', 'Loss-of-separation pair-steps per episode', '{:.2f}'),
                              (axes[1], 'edges_mean', 'Active conflict edges per step', '{:.3f}')):
    labels, vals, errs, cols = [], [], [], []
    for st in (2, 3):
        for m in ORDER[st]:
            x = ev[(ev.stage == st) & (ev.method == m)]
            if st == 2:
                g = x.groupby('seed')[metric].mean(); v, e_ = g.mean(), g.std(ddof=1)
            else:
                v, e_ = x[metric].mean(), 1.96 * x[metric].std(ddof=1) / math.sqrt(len(x))
            labels.append('Stage {}\n{}'.format(st, NAME_WRAP[m])); vals.append(v); errs.append(e_); cols.append(MCOL[m])
    xx = np.arange(len(vals))
    ax.bar(xx, vals, 0.6, yerr=errs, color=cols, edgecolor='white', linewidth=2, capsize=3, error_kw=dict(lw=1))
    for xi, v, e_ in zip(xx, vals, errs):
        ax.text(xi, v + e_ + max(vals) * 0.02, fmt.format(v), ha='center', va='bottom', fontsize=8.2)
    ax.axvline(2.5, color='#bbb', lw=1, ls=':')
    ax.set_xticks(xx); ax.set_xticklabels(labels, fontsize=7.6); ax.set_ylabel(ylab)
    ax.set_ylim(0, max(v + e_ for v, e_ in zip(vals, errs)) * 1.25)
fig.suptitle('IGAT-MARL metrics check (evaluation pass). Stage 2: mean ± std over 5 seeds; Stage 3: mean ± 95% CI over 200 episodes', fontsize=10)
plt.tight_layout(rect=(0, 0, 1, 0.93)); save(fig, 'igat_marl_metrics_check')

# ---- Advantage-correlation check ------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(14, 5.4))
specs = (('corr_m_s', 'corr(mission advantage,\nsafety advantage)', (-1, 1)),
         ('cos_own_05', 'cosine(learning signal with own α,\nlearning signal with α = 0.5)', (-1, 1)),
         ('corr_own_comb', 'corr(learning signal with own α,\nplain combined-reward advantage)', (-1, 1)))
for ax, (metric, ylab, ylim) in zip(axes, specs):
    labels, vals, errs, cols = [], [], [], []
    for st in (2, 3):
        for m in ORDER[st]:
            x = fin[(fin.stage == st) & (fin.method == m)]
            g = x.groupby('seed')[metric].mean() if st == 2 else x[metric]
            labels.append('Stage {}\n{}'.format(st, NAME_WRAP[m])); vals.append(g.mean()); errs.append(g.std(ddof=1)); cols.append(MCOL[m])
    xx = np.arange(len(vals))
    ax.bar(xx, vals, 0.6, yerr=errs, color=cols, edgecolor='white', linewidth=2, capsize=3, error_kw=dict(lw=1))
    for xi, v, e_ in zip(xx, vals, errs):
        ax.text(xi, v + (e_ if v >= 0 else -e_) + (0.04 if v >= 0 else -0.04), '{:.2f}'.format(v),
                ha='center', va='bottom' if v >= 0 else 'top', fontsize=8)
    ax.axhline(0, color='#444', lw=0.8); ax.axvline(2.5, color='#bbb', lw=1, ls=':')
    flat = [l.replace('\n', ' ') for l in labels]
    ax.set_xticks(xx); ax.set_xticklabels(flat, fontsize=7.6, rotation=35, ha='right'); ax.set_ylabel(ylab, fontsize=8.8); ax.set_ylim(*ylim)
fig.suptitle('Advantage-correlation check (final checkpoints; 10 training-style batches each; Stage 2 error bars = std over 5 seeds, '
             'Stage 3 = std over batches)', fontsize=10)
plt.tight_layout(rect=(0, 0, 1, 0.93)); save(fig, 'advantage_correlation_check')
print('saved: collision_type_check, safety_reward_frequency_check, alpha_spread_check, igat_marl_metrics_check, advantage_correlation_check')

saved: collision_type_check, safety_reward_frequency_check, alpha_spread_check, igat_marl_metrics_check, advantage_correlation_check


In [9]:
# Cell 9 -- summary.json (the README quotes these)
summary = {'collision_type_check': T1.to_dict(orient='records'),
           'igat_marl_metrics_check': T4.to_dict(orient='records'),
           'safety_reward_frequency_and_alpha_spread': T23.to_dict(orient='records'),
           'alpha_vs_target_distance': ad.assign(method=ad.method.map(NAME)).round(4).to_dict(orient='records'),
           'advantage_correlation_check_final': TA.to_dict(orient='records')}
json.dump(summary, open(OUT + '/summary.json', 'w'), indent=1, default=str, ensure_ascii=False)
print('saved', OUT + '/summary.json')

saved output/summary.json
